# Customer Usage Analysis
This notebook explores customer usage patterns and behaviors in our telecom dataset. We'll analyze call patterns, service plans, and customer service interactions to understand our customer base better.

**Note:** This notebook automatically loads data from Athena when available, or uses sample data as a fallback.

## Setup: Load Data with Fallback
This cell attempts to load data from the Athena table `sagemaker_sample_db.churn`. If the table is unavailable (doesn't exist, permissions issues, etc.), it automatically uses fallback sample data.

**Run this cell first before executing any SQL queries.**

In [0]:
import pandas as pd
import numpy as np
import warnings

# =============================================================================
# FALLBACK DATA DEFINITION
# =============================================================================

def get_fallback_churn_data() -> pd.DataFrame:
    """
    Returns a hardcoded sample of churn data that mirrors the structure
    of the sagemaker_sample_db.churn table.
    """
    np.random.seed(42)
    n_samples = 500
    
    states = ['KS', 'OH', 'NJ', 'OK', 'AL', 'MA', 'MO', 'LA', 'WV', 'IN',
              'RI', 'IA', 'MT', 'NY', 'ID', 'VT', 'VA', 'TX', 'FL', 'CO',
              'AZ', 'CA', 'WA', 'OR', 'NV', 'UT', 'NM', 'GA', 'NC', 'SC']
    
    data = {
        'state': np.random.choice(states, n_samples),
        'account_length': np.random.randint(1, 250, n_samples),
        'area_code': np.random.choice([408, 415, 510], n_samples),
        'phone': [f'{np.random.randint(100,999)}-{np.random.randint(1000,9999)}' for _ in range(n_samples)],
        'intl_plan': np.random.choice(['yes', 'no'], n_samples, p=[0.1, 0.9]),
        'vmail_plan': np.random.choice(['yes', 'no'], n_samples, p=[0.3, 0.7]),
        'vmail_message': np.random.randint(0, 50, n_samples),
        'day_mins': np.round(np.random.uniform(0, 350, n_samples), 1),
        'day_calls': np.random.randint(0, 165, n_samples),
        'day_charge': np.round(np.random.uniform(0, 60, n_samples), 2),
        'eve_mins': np.round(np.random.uniform(0, 360, n_samples), 1),
        'eve_calls': np.random.randint(0, 170, n_samples),
        'eve_charge': np.round(np.random.uniform(0, 30, n_samples), 2),
        'night_mins': np.round(np.random.uniform(0, 400, n_samples), 1),
        'night_calls': np.random.randint(0, 175, n_samples),
        'night_charge': np.round(np.random.uniform(0, 18, n_samples), 2),
        'intl_mins': np.round(np.random.uniform(0, 20, n_samples), 1),
        'intl_calls': np.random.randint(0, 20, n_samples),
        'intl_charge': np.round(np.random.uniform(0, 5.5, n_samples), 2),
        'custserv_calls': np.random.randint(0, 10, n_samples),
        'churn': np.random.choice(['True.', 'False.'], n_samples, p=[0.15, 0.85])
    }
    
    return pd.DataFrame(data)


def print_fallback_warning(error, resolution_steps):
    """
    Prints a formatted warning message when falling back to sample data.
    """
    print("\n" + "="*80)
    print("⚠️  FALLBACK MODE ACTIVATED")
    print("="*80)
    print(f"\n❌ Failed to load data from Athena table: sagemaker_sample_db.churn")
    print(f"\n📋 Error Details: {type(error).__name__}: {str(error)[:200]}")
    print("\n✅ Using fallback sample data. All SQL queries will work correctly.")
    print("\n🔧 To use real Athena data, address the following:")
    for i, step in enumerate(resolution_steps, 1):
        print(f"   {i}. {step}")
    print("\n📝 After fixing the issue, re-run this cell to load real data.")
    print("="*80 + "\n")


# =============================================================================
# ATTEMPT TO LOAD DATA FROM ATHENA
# =============================================================================

USE_FALLBACK = False
churn = None  # This will be the DataFrame used by SQL cells

print("🔍 Attempting to load data from Athena table 'sagemaker_sample_db.churn'...\n")

try:
    import boto3
    import time
    import io
    
    # Initialize clients
    athena = boto3.client('athena')
    s3 = boto3.client('s3')
    sts = boto3.client('sts')
    
    account_id = sts.get_caller_identity()['Account']
    region = boto3.Session().region_name
    
    # Get workgroup output location
    try:
        workgroup = athena.get_work_group(WorkGroup='primary')
        output_location = workgroup['WorkGroup']['Configuration']['ResultConfiguration']['OutputLocation']
    except:
        output_location = f's3://aws-athena-query-results-{account_id}-{region}/'
    
    # Query to load all data from the churn table
    query = "SELECT * FROM sagemaker_sample_db.churn"
    
    response = athena.start_query_execution(
        QueryString=query,
        ResultConfiguration={'OutputLocation': output_location}
    )
    
    query_execution_id = response['QueryExecutionId']
    
    # Wait for query to complete
    print("   Executing Athena query...")
    max_attempts = 60
    for attempt in range(max_attempts):
        result = athena.get_query_execution(QueryExecutionId=query_execution_id)
        state = result['QueryExecution']['Status']['State']
        
        if state == 'SUCCEEDED':
            break
        elif state in ['FAILED', 'CANCELLED']:
            error_msg = result['QueryExecution']['Status'].get('StateChangeReason', 'Unknown error')
            raise Exception(f"Query {state}: {error_msg}")
        
        time.sleep(1)
    else:
        raise Exception("Query timed out after 60 seconds")
    
    # Get the S3 location of results
    output_location = result['QueryExecution']['ResultConfiguration']['OutputLocation']
    
    # Parse S3 path
    s3_path = output_location.replace('s3://', '')
    bucket = s3_path.split('/')[0]
    key = '/'.join(s3_path.split('/')[1:])
    
    # Read CSV from S3
    print("   Downloading results from S3...")
    response = s3.get_object(Bucket=bucket, Key=key)
    churn = pd.read_csv(io.BytesIO(response['Body'].read()))
    
    print(f"\n✅ Successfully loaded {len(churn)} rows from Athena!")
    print(f"   Source: sagemaker_sample_db.churn")
    print(f"   Columns: {len(churn.columns)}")
    USE_FALLBACK = False

except Exception as e:
    USE_FALLBACK = True
    churn = get_fallback_churn_data()
    
    error_str = str(e).lower()
    
    if 'table' in error_str or 'database' in error_str or 'not found' in error_str or 'does not exist' in error_str:
        resolution_steps = [
            "Verify the database 'sagemaker_sample_db' exists in AWS Glue Data Catalog",
            "Check that the 'churn' table exists in the database",
            "Run the data ingestion notebook to create the table",
            "Ensure you're connected to the correct AWS account and region"
        ]
    elif 'access' in error_str or 'permission' in error_str or 'denied' in error_str:
        resolution_steps = [
            "Check IAM permissions for Athena query execution (athena:StartQueryExecution, athena:GetQueryExecution)",
            "Verify Glue Data Catalog permissions (glue:GetTable, glue:GetDatabase)",
            "Ensure S3 read permissions for the underlying data location",
            "Check S3 write permissions for Athena query results bucket",
            "Contact your AWS administrator for permission grants"
        ]
    elif 'workgroup' in error_str or 'output' in error_str or 'location' in error_str:
        resolution_steps = [
            "Configure an Athena workgroup with a valid S3 output location",
            "Ensure the S3 bucket for query results exists",
            "Check S3 write permissions for the results bucket",
            "Verify the primary workgroup configuration in Athena console"
        ]
    elif 'timeout' in error_str:
        resolution_steps = [
            "The query took too long to execute",
            "Check if the table has a large amount of data",
            "Verify Athena service is responding normally",
            "Try running a simple query in the Athena console"
        ]
    else:
        resolution_steps = [
            "Verify AWS credentials are configured correctly",
            "Check network connectivity to AWS services",
            "Ensure the sagemaker_sample_db.churn table exists",
            "Review IAM permissions for Athena, Glue, and S3",
            f"Error type: {type(e).__name__}"
        ]
    
    print_fallback_warning(e, resolution_steps)

# =============================================================================
# SUMMARY
# =============================================================================

print("\n" + "-"*60)
print("📊 DATA LOADED AND READY FOR QUERIES")
print("-"*60)
print(f"   DataFrame name: churn")
print(f"   Rows: {len(churn)}")
print(f"   Columns: {list(churn.columns)}")
print(f"   Data source: {'Athena (sagemaker_sample_db.churn)' if not USE_FALLBACK else 'Fallback sample data'}")
print("-"*60)
print("\n✅ You can now run the SQL cells below. They will query the 'churn' DataFrame.")

## Dataset Overview
Our analysis uses the churn dataset containing customer information including:

- Geographic data (state, area_code)
- Account details (account_length, phone)
- Service plans (intl_plan, vmail_plan)
- Usage metrics across different time periods (day, evening, night, international)
- Customer service interactions

### Query 1: Data Preview
Get familiar with the data structure and see actual values in each column by examining the structure and sample data from our table.

In [0]:

_results = []
_stream = sqlutils.sql_stream("-- Preview the dataset to understand our data structure\n-- Note: Queries the 'churn' DataFrame loaded in the setup cell\nSELECT *\nFROM churn\nLIMIT 10")
for result in _stream:
    if result.status == "success":
        display(result.result)
        _results.append(result)
    else:
        # Save partial results for debugging before raising
        if len(_results) > 0:
            for r in _results:
                globals()[f'sql_output_6kzw_{r.statement_index}'] = r.result
        raise Exception(result.error)

# All statements succeeded - save indexed vars if multiple results
if len(_results) > 1:
    for r in _results:
        globals()[f'sql_output_6kzw_{r.statement_index}'] = r.result

# Always save main variable
sql_output_6kzw = _results[0].result if len(_results) == 1 else [r.result for r in _results]


## Query 2: Geographic Distribution Analysis
Identify our key markets and see if there are regional differences in usage patterns and customer loyalty (account length) to help with regional planning and marketing strategies.

In [0]:

_results = []
_stream = sqlutils.sql_stream("-- Analyze customer distribution by state and area code\nSELECT \n    state,\n    area_code,\n    COUNT(*) as customer_count,\n    ROUND(AVG(account_length), 1) as avg_account_length_months,\n    ROUND(AVG(day_mins + eve_mins + night_mins), 2) as avg_total_minutes\nFROM churn\nGROUP BY state, area_code\nORDER BY customer_count DESC\nLIMIT 15")
for result in _stream:
    if result.status == "success":
        display(result.result)
        _results.append(result)
    else:
        # Save partial results for debugging before raising
        if len(_results) > 0:
            for r in _results:
                globals()[f'sql_output_bf8q_{r.statement_index}'] = r.result
        raise Exception(result.error)

# All statements succeeded - save indexed vars if multiple results
if len(_results) > 1:
    for r in _results:
        globals()[f'sql_output_bf8q_{r.statement_index}'] = r.result

# Always save main variable
sql_output_bf8q = _results[0].result if len(_results) == 1 else [r.result for r in _results]


## Query 3: Service Plan Adoption and Usage Patterns
Understand the relationship between service plans and actual usage to identify upselling opportunities and plan effectiveness by examining how different service plans correlate with usage behavior.

In [0]:

_results = []
_stream = sqlutils.sql_stream("-- Compare usage patterns between international and voicemail plan subscribers\nSELECT \n    intl_plan,\n    vmail_plan,\n    COUNT(*) as customers,\n    ROUND(AVG(intl_mins), 2) as avg_intl_minutes,\n    ROUND(AVG(intl_charge), 2) as avg_intl_charges,\n    ROUND(AVG(vmail_message), 1) as avg_voicemail_messages,\n    ROUND(AVG(day_mins + eve_mins + night_mins), 2) as avg_total_voice_minutes\nFROM churn\nGROUP BY intl_plan, vmail_plan\nORDER BY customers DESC")
for result in _stream:
    if result.status == "success":
        display(result.result)
        _results.append(result)
    else:
        # Save partial results for debugging before raising
        if len(_results) > 0:
            for r in _results:
                globals()[f'sql_output_5sbf_{r.statement_index}'] = r.result
        raise Exception(result.error)

# All statements succeeded - save indexed vars if multiple results
if len(_results) > 1:
    for r in _results:
        globals()[f'sql_output_5sbf_{r.statement_index}'] = r.result

# Always save main variable
sql_output_5sbf = _results[0].result if len(_results) == 1 else [r.result for r in _results]


## Query 4: Comprehensive Customer Profile Analysis
Combine geographic distribution, service plan adoption, and usage timing patterns to create comprehensive customer segments that reveal actionable business insights.

In [0]:

_results = []
_stream = sqlutils.sql_stream("-- Create comprehensive customer segments combining geography, service plans, and usage patterns\nSELECT \n    state,\n    intl_plan,\n    vmail_plan,\n    CASE \n        WHEN day_mins > eve_mins AND day_mins > night_mins THEN 'Day Heavy'\n        WHEN eve_mins > day_mins AND eve_mins > night_mins THEN 'Evening Heavy' \n        WHEN night_mins > day_mins AND night_mins > eve_mins THEN 'Night Heavy'\n        ELSE 'Balanced'\n    END as usage_pattern,\n    COUNT(*) as customer_count,\n    ROUND(AVG(account_length), 1) as avg_account_length_months,\n    ROUND(AVG(day_charge + eve_charge + night_charge + intl_charge), 2) as avg_total_charges,\n    ROUND(AVG(intl_mins), 2) as avg_intl_minutes,\n    ROUND(AVG(vmail_message), 1) as avg_voicemail_messages,\n    ROUND(AVG(custserv_calls), 2) as avg_customer_service_calls,\n    ROUND(AVG(day_mins + eve_mins + night_mins + intl_mins), 2) as avg_total_minutes\nFROM churn\nGROUP BY state, intl_plan, vmail_plan, usage_pattern\nHAVING COUNT(*) >= 5\nORDER BY customer_count DESC")
for result in _stream:
    if result.status == "success":
        display(result.result)
        _results.append(result)
    else:
        # Save partial results for debugging before raising
        if len(_results) > 0:
            for r in _results:
                globals()[f'sql_output_ag80_{r.statement_index}'] = r.result
        raise Exception(result.error)

# All statements succeeded - save indexed vars if multiple results
if len(_results) > 1:
    for r in _results:
        globals()[f'sql_output_ag80_{r.statement_index}'] = r.result

# Always save main variable
sql_output_ag80 = _results[0].result if len(_results) == 1 else [r.result for r in _results]


## Execution Summary

In [0]:
# Print execution summary
print("="*60)
print("📊 EXECUTION SUMMARY")
print("="*60)

if USE_FALLBACK:
    print("\n⚠️  Data Source: FALLBACK SAMPLE DATA")
    print("\n   The Athena table 'sagemaker_sample_db.churn' was not available.")
    print("   All queries ran successfully using synthetic sample data that")
    print("   mirrors the structure and characteristics of the real dataset.")
    print("\n🔧 To use real Athena data:")
    print("   1. Ensure the database 'sagemaker_sample_db' exists in Glue Data Catalog")
    print("   2. Create the 'churn' table with the expected schema")
    print("   3. Verify IAM permissions for Athena and Glue access")
    print("   4. Re-run the Setup cell at the top of this notebook")
else:
    print("\n✅ Data Source: ATHENA (sagemaker_sample_db.churn)")
    print("\n   All queries executed successfully using production data.")
    print(f"   Total rows in dataset: {len(churn)}")

print("\n" + "="*60)

## Key Insights Expected
From this analysis, we expect to uncover:

- **Geographic Hotspots**: Which states and area codes have the highest customer concentration
- **Plan Effectiveness**: How well our international and voicemail plans align with actual usage
- **Usage Timing**: Customer preferences for different time periods and network load distribution
- **Customer Service Patterns**: Whether certain usage behaviors correlate with higher support needs
- **Comprehensive Segments**: Complete customer profiles combining geography, plans, and usage patterns

This analysis will help inform marketing strategies, network planning, customer service optimization, and retention efforts.

---

**Note**: Check the Execution Summary above to see whether results are from the Athena table or fallback sample data.